[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdsanch1/simrc/blob/master/01.%20Parte%201/03.%20Clase%203/03Class%20NB.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jdsanch1/SimRC/master?labpath=01.%20Parte%201%2F03.%20Clase%203%2F03Class%20NB.ipynb)

In [ ]:
import importlib, subprocess, sys

_required = ["yfinance", "pandas", "numpy", "matplotlib", "seaborn", "scipy", "sklearn", "statsmodels"]
_missing  = [pkg for pkg in _required if importlib.util.find_spec(pkg) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _missing)

# Clase 3: Opciones financieras — payoff, P&L y volatilidad implícita

**[Juan Diego Sánchez Torres](https://www.researchgate.net/profile/Juan_Diego_Sanchez_Torres)**  
*Profesor*, [MAF ITESO](http://maf.iteso.mx/web/general/detalle?group_id=5858156)  
dsanchez@iteso.mx

## Objetivos de aprendizaje

- Descargar cadenas de opciones reales con `yfinance`.
- Calcular y graficar el **payoff** y **P&L** de calls y puts.
- Entender la **volatilidad implícita** y el fenómeno de *smile/skew*.
- Verificar empíricamente la **paridad put-call**.
- Conocer los fundamentos del modelo de **Black-Scholes-Merton**.

---

## Introducción teórica

Una **opción financiera** es un contrato que otorga al comprador el *derecho* (pero no la obligación) de comprar o vender un activo subyacente a un precio predeterminado (**strike**, $K$) en una fecha futura (**vencimiento**, $T$).

### Tipos básicos

| Tipo | Derecho del comprador | Payoff a vencimiento |
|------|----------------------|---------------------|
| **Call** | Comprar el activo a precio $K$ | $\max(S_T - K, \, 0)$ |
| **Put** | Vender el activo a precio $K$ | $\max(K - S_T, \, 0)$ |

donde $S_T$ es el precio del activo al vencimiento.

### Participantes

- **Comprador** (*holder*): paga la **prima** $c$ (call) o $p$ (put) y adquiere el derecho.
- **Vendedor** (*writer*): recibe la prima y asume la obligación.

### Ganancia y pérdida (P&L)

| Posición | Call | Put |
|----------|------|-----|
| **Comprador** | $\max(S_T - K, 0) - c$ | $\max(K - S_T, 0) - p$ |
| **Vendedor** | $c - \max(S_T - K, 0)$ | $p - \max(K - S_T, 0)$ |

El P&L es un **juego de suma cero**: lo que gana el comprador lo pierde el vendedor, y viceversa (Hull, 2018, Cap. 1; Venegas Martínez, 2008, Cap. 5).

## 1. Descarga de datos con `yfinance`

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# Estilo visual
sns.set_theme(style="darkgrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 100, "figure.figsize": (12, 5)})
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 10)

Descargamos precios ajustados de AAPL para contextualizar el análisis de opciones.

> **Nota BMV**: Para acciones de la Bolsa Mexicana de Valores, el ticker incluye `.MX`.  
> Ejemplo: `MEXCHEM.MX`, `GFINBURO.MX`, `GFNORTEO.MX`.

In [ ]:
tickers    = ["AA", "AAPL", "MSFT", "^GSPC"]
start_date = "2025-01-01"
end_date   = "2025-03-27"

data = yf.download(tickers, start=start_date, end=end_date,
                   auto_adjust=True, progress=False)
closes = data["Close"]
closes.tail()

---
## 2. Cadena de opciones con `yfinance`

### Descarga de opciones reales

`yfinance` permite descargar la **cadena de opciones** (*option chain*) de cualquier ticker. El objeto `Ticker` provee:
- `.options`: lista de fechas de vencimiento disponibles.
- `.option_chain(date)`: devuelve un objeto con `.calls` y `.puts` (DataFrames).

In [ ]:
aapl = yf.Ticker("AAPL")

# Fechas de vencimiento disponibles
print("Fechas de vencimiento disponibles:")
print(aapl.options[:8], "...")

In [ ]:
# Seleccionar el vencimiento más cercano
expiry = aapl.options[0]
print(f"Vencimiento seleccionado: {expiry}")

opt_chain = aapl.option_chain(expiry)
print(f"\nCalls: {len(opt_chain.calls)} contratos")
print(f"Puts:  {len(opt_chain.puts)} contratos")

In [ ]:
# Cadena de calls
opt_chain.calls[["strike", "lastPrice", "bid", "ask", "volume", "openInterest", "impliedVolatility"]].head(10)

In [ ]:
# Cadena de puts
opt_chain.puts[["strike", "lastPrice", "bid", "ask", "volume", "openInterest", "impliedVolatility"]].head(10)

---
## 3. Volatilidad implícita

### Concepto

La **volatilidad implícita** (IV) es el valor de $\sigma$ que, al sustituirlo en la fórmula de Black-Scholes, reproduce el precio de mercado de la opción. Es una medida del **consenso del mercado** sobre la volatilidad futura del activo.

Para una call europea, el precio teórico de Black-Scholes es:

$$
C = S_0 \, \Phi(d_1) - K \, e^{-rT} \, \Phi(d_2)
$$

donde:

$$
d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}
$$

y $\Phi$ es la función de distribución acumulada de la normal estándar.

La IV se obtiene invirtiendo numéricamente esta ecuación: dado el precio de mercado $C^{\text{mkt}}$, encontrar $\sigma^{\text{impl}}$ tal que $C(\sigma^{\text{impl}}) = C^{\text{mkt}}$.

### Volatility smile y skew

Si Black-Scholes fuera exacto (rendimientos log-normales), la IV sería **constante** para todos los strikes. En la práctica se observa:
- **Smile**: IV mínima at-the-money (ATM), mayor para opciones in-the-money (ITM) y out-of-the-money (OTM).
- **Skew**: IV más alta para puts OTM que para calls OTM, reflejando la mayor demanda de protección ante caídas (Hull, 2018, Cap. 20).

In [ ]:
# Precio actual del subyacente
spot = closes["AAPL"].iloc[-1]
print(f"Precio spot AAPL: ${spot:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Calls IV
calls = opt_chain.calls[opt_chain.calls["impliedVolatility"] > 0.001].copy()
axes[0].plot(calls["strike"], calls["impliedVolatility"], "o-", markersize=4)
axes[0].axvline(spot, color="green", linestyle="--", label=f"Spot = ${spot:.0f}")
axes[0].set_title(f"Volatilidad implícita — Calls ({expiry})")
axes[0].set_xlabel("Strike (K)")
axes[0].set_ylabel("IV")
axes[0].legend()

# Puts IV
puts = opt_chain.puts[opt_chain.puts["impliedVolatility"] > 0.001].copy()
axes[1].plot(puts["strike"], puts["impliedVolatility"], "o-", markersize=4, color="tab:orange")
axes[1].axvline(spot, color="green", linestyle="--", label=f"Spot = ${spot:.0f}")
axes[1].set_title(f"Volatilidad implícita — Puts ({expiry})")
axes[1].set_xlabel("Strike (K)")
axes[1].set_ylabel("IV")
axes[1].legend()

plt.suptitle("Volatility smile / skew", fontsize=14)
plt.tight_layout()

---
## 4. Payoff de opciones

### Definición formal

El **payoff** es el valor intrínseco de la opción al vencimiento, **antes** de descontar la prima pagada.

$$
\text{Payoff}_{\text{call}} = \max(S_T - K, \, 0) = (S_T - K)^+
$$

$$
\text{Payoff}_{\text{put}} = \max(K - S_T, \, 0) = (K - S_T)^+
$$

La notación $(x)^+ = \max(x, 0)$ es estándar en la literatura de derivados.

In [ ]:
def option_payoff(ST, K, option_type="call"):
    """Payoff de una opción europea a vencimiento."""
    if option_type == "call":
        return np.maximum(ST - K, 0)
    else:
        return np.maximum(K - ST, 0)

# Rango de precios a vencimiento
ST = np.linspace(50, 300, 500)
K = 170  # Strike de ejemplo

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ST, option_payoff(ST, K, "call"), linewidth=2, color="tab:blue")
axes[0].axhline(0, color="black", linewidth=0.5)
axes[0].axvline(K, color="gray", linestyle="--", alpha=0.5, label=f"K = {K}")
axes[0].set_title(f"Payoff — Call (K = {K})")
axes[0].set_xlabel("$S_T$ (precio a vencimiento)")
axes[0].set_ylabel("Payoff")
axes[0].legend()

axes[1].plot(ST, option_payoff(ST, K, "put"), linewidth=2, color="tab:orange")
axes[1].axhline(0, color="black", linewidth=0.5)
axes[1].axvline(K, color="gray", linestyle="--", alpha=0.5, label=f"K = {K}")
axes[1].set_title(f"Payoff — Put (K = {K})")
axes[1].set_xlabel("$S_T$ (precio a vencimiento)")
axes[1].set_ylabel("Payoff")
axes[1].legend()

plt.tight_layout()

---
## 5. Ganancia y pérdida (P&L)

El **P&L** incorpora la prima pagada (o recibida). Es la ganancia neta real de la posición.

Para el **comprador** de una call con prima $c$:

$$
\text{P\&L}_{\text{comprador}} = \max(S_T - K, 0) - c
$$

Para el **vendedor**:

$$
\text{P\&L}_{\text{vendedor}} = c - \max(S_T - K, 0)
$$

Nótese que $\text{P\&L}_{\text{comprador}} + \text{P\&L}_{\text{vendedor}} = 0$: es un **juego de suma cero**.

In [ ]:
def option_pnl(ST, K, premium, option_type="call", position="buyer"):
    """P&L de una posición en opciones."""
    payoff = option_payoff(ST, K, option_type)
    if position == "buyer":
        return payoff - premium
    else:
        return premium - payoff

K = 170
premium_call = 8
premium_put = 5

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

configs = [
    ("call", "buyer",  premium_call, "Call — Comprador", "tab:blue"),
    ("call", "seller", premium_call, "Call — Vendedor",  "tab:red"),
    ("put",  "buyer",  premium_put,  "Put — Comprador",  "tab:blue"),
    ("put",  "seller", premium_put,  "Put — Vendedor",   "tab:red"),
]

for ax, (otype, pos, prem, title, color) in zip(axes.flat, configs):
    pnl = option_pnl(ST, K, prem, otype, pos)
    ax.plot(ST, pnl, linewidth=2, color=color)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.axvline(K, color="gray", linestyle="--", alpha=0.5)
    ax.fill_between(ST, pnl, 0, where=(pnl > 0), alpha=0.15, color="green")
    ax.fill_between(ST, pnl, 0, where=(pnl < 0), alpha=0.15, color="red")
    ax.set_title(f"{title}  (K={K}, prima={prem})")
    ax.set_xlabel("$S_T$")
    ax.set_ylabel("P&L")

plt.suptitle("Ganancia y pérdida de opciones europeas", fontsize=14)
plt.tight_layout()

---
## 6. P&L combinado: comprador vs. vendedor

El siguiente gráfico superpone las posiciones del comprador y vendedor para visualizar la simetría del juego de suma cero.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, otype, prem, title in [
    (axes[0], "call", premium_call, "Call"),
    (axes[1], "put",  premium_put,  "Put")
]:
    pnl_buyer  = option_pnl(ST, K, prem, otype, "buyer")
    pnl_seller = option_pnl(ST, K, prem, otype, "seller")
    ax.plot(ST, pnl_buyer,  label="Comprador", linewidth=2)
    ax.plot(ST, pnl_seller, label="Vendedor",  linewidth=2, linestyle="--")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.axvline(K, color="gray", linestyle="--", alpha=0.5)
    ax.set_title(f"P&L {title} (K={K}, prima={prem})")
    ax.set_xlabel("$S_T$")
    ax.set_ylabel("P&L")
    ax.legend()

plt.tight_layout()

---
## 7. Paridad put-call

Para opciones europeas sobre un activo que no paga dividendos, existe una relación fundamental entre el precio de una call y una put con el mismo strike y vencimiento:

$$
C - P = S_0 - K \, e^{-rT}
$$

Esta relación, conocida como **paridad put-call**, implica que conocer el precio de una call determina el precio de la put (y viceversa). Es una condición de **no arbitraje** (Hull, 2018, Cap. 11; Venegas Martínez, 2008, Cap. 6).

### Verificación empírica

Podemos verificar esta relación con los datos de mercado descargados:

In [ ]:
# Verificar paridad put-call con datos reales
r_free = 0.045  # Tasa libre de riesgo aproximada (Fed Funds 2025)

# Emparejar calls y puts por strike
merged = pd.merge(
    opt_chain.calls[["strike", "lastPrice"]].rename(columns={"lastPrice": "C"}),
    opt_chain.puts[["strike", "lastPrice"]].rename(columns={"lastPrice": "P"}),
    on="strike"
)

# Calcular ambos lados de la paridad
import datetime
T = (pd.Timestamp(expiry) - pd.Timestamp.now()).days / 365
merged["C - P"] = merged["C"] - merged["P"]
merged["S - K*exp(-rT)"] = spot - merged["strike"] * np.exp(-r_free * T)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(merged["S - K*exp(-rT)"], merged["C - P"], alpha=0.5, s=20)
lims = [merged[["C - P", "S - K*exp(-rT)"]].min().min(),
        merged[["C - P", "S - K*exp(-rT)"]].max().max()]
ax.plot(lims, lims, "r--", label="Paridad perfecta")
ax.set_xlabel("$S_0 - K \cdot e^{-rT}$")
ax.set_ylabel("$C - P$")
ax.set_title("Verificación de la paridad put-call")
ax.legend()
plt.tight_layout()

---

## Navegación del curso

← **Anterior**: Clase 2: Retornos y covarianza  
→ **Siguiente**: Clase 4: Ratio de Sharpe y portafolio óptimo

---

## 8. Referencias bibliográficas

- **Black, F. & Scholes, M.** (1973). The Pricing of Options and Corporate Liabilities. *Journal of Political Economy*, 81(3), 637–654.
- **Cont, R.** (2001). Empirical Properties of Asset Returns: Stylized Facts and Statistical Issues. *Quantitative Finance*, 1(2), 223–236.
- **Hull, J. C.** (2018). *Options, Futures, and Other Derivatives* (10th ed.). Pearson.
  - Cap. 1: Introduction. Cap. 10–11: Mechanics of Options Markets, Properties of Stock Options. Cap. 15: The Black-Scholes-Merton Model. Cap. 20: Volatility Smiles.
- **Luenberger, D. G.** (2013). *Investment Science* (2nd ed.). Oxford University Press. — Cap. 12–13: Basic Options Theory.
- **Merton, R. C.** (1973). Theory of Rational Option Pricing. *The Bell Journal of Economics and Management Science*, 4(1), 141–183.
- **Tsay, R. S.** (2010). *Analysis of Financial Time Series* (3rd ed.). Wiley.
- **Venegas Martínez, F.** (2008). *Riesgos financieros y económicos: productos derivados y decisiones económicas bajo incertidumbre* (2a ed.). Cengage Learning. — Cap. 5–6: Opciones, paridad put-call, Black-Scholes.